<a href="https://colab.research.google.com/github/pokechiu827/CSC2053MyCode/blob/main/HW5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW5 - Lunar Lander Reinforcement Learning

For this homework we will be applying reinforcement learning to the lunar lander environment.

In [1]:
!pip install swig
!pip install gymnasium[box2d]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 92.9 MB/s eta 0:00:00


In [2]:
import tensorflow as tf
import numpy as np
import matplotlib.animation
import matplotlib.pyplot as plt
plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)
plt.rc('animation', html='jshtml')
import gymnasium as gym

In [ ]:
env = gym.make('LunarLander-v3', render_mode="rgb_array")
obs, info = env.reset()

def plot_environment(env, figsize=(5, 4)):
    plt.figure(figsize=figsize)
    img = env.render()
    plt.imshow(img)
    plt.axis("off")
    return img

plot_environment(env)
plt.show()

In [ ]:
print("Lander coordinates: ", obs[0], obs[1])
print("Lander velocities: ", obs[2], obs[3])
print("Lander angle: ", obs[4])
print("Lander angular velocity: ", obs[5])
print("Legs touching ground?: ", obs[6], obs[7])

### Action Space:

0. do nothing
1. fire left engine
2. fire main engine
3. fire right engine

In [ ]:
for x in range(25):
    obs, reward, done, truncated, info = env.step(0)
plot_environment(env)
plt.show()
print("Lander coordinates: ", obs[0], obs[1])
print("Lander velocities: ", obs[2], obs[3])
print("Lander angle: ", obs[4])
print("Lander angular velocity: ", obs[5])
print("Legs touching ground?: ", obs[6], obs[7])
print("Reward: ", reward)

### Write a better basic_policy than the one below.  

Rewards are given for:
1. Distance to landing pad (closer is better)
2. Speed of lander (slower is better)
3. Penalty for how far the lander is tilted
4. +10 pts for each leg touching ground
5. -0.03 pts for each step in which a side engine is firing
6. -0.3 pts for each step in which the main engine is firing
7. -100 for crashing
8. +100 for landing safely

Full credit is given for a basic policy which achieves a mean total > 0.  Partial credit given for basic policies which achieve mean scores below 0.

In [ ]:
def basic_policy(obs):
    x, y, vx, vy, ang, angV, lleg, rleg = obs
    # Write some rules here to achieve better performance for your lander

    return 0


totals = []
for episode in range(100):
    episode_rewards = 0
    obs, info = env.reset(seed=episode)
    for step in range(200):
        action = basic_policy(obs)
        obs, reward, done, truncated, info = env.step(action)
        episode_rewards += reward
        if done or truncated:
            break

    totals.append(episode_rewards)

print("Mean total score: ", np.mean(totals))

def update_scene(num, frames, patch):
    patch.set_data(frames[num])
    return patch,

def plot_animation(frames, repeat=False, interval=40):
    fig = plt.figure()
    patch = plt.imshow(frames[0])
    plt.axis('off')
    anim = matplotlib.animation.FuncAnimation(
        fig, update_scene, fargs=(frames, patch),
        frames=len(frames), repeat=repeat, interval=interval)
    plt.close()
    return anim

def show_one_episode(policy, n_max_steps=200):
    frames = []
    env = gym.make("LunarLander-v3", render_mode="rgb_array")
    obs, info = env.reset()
    score = 0
    for step in range(n_max_steps):
        frames.append(env.render())
        action = policy(obs)
        obs, reward, done, truncated, info = env.step(action)
        score += reward
        if done or truncated:
            break
    env.close()
    return plot_animation(frames)

show_one_episode(basic_policy)

In [ ]:
# If you want to look at the values after every 5 actions, use this cell instead
env = gym.make("LunarLander-v3", render_mode="rgb_array")
np.random.seed(42)
obs, info = env.reset(seed=42)
for step in range(200):
    if step%5==0:
        plot_environment(env)
        plt.show()
        print(obs)
    action = basic_policy(obs)
    obs, reward, done, truncated, info = env.step(action)
    if done or truncated:
        break
env.close()

## Modify the CartPole Example

Use the code below (copied from the CartPole example in class) as your starting point.  Recall that the CartPole environment only had 2 actions (push left or push right), whereas our current problem has 4 possible actions (listed above).

1. Modify the neural architecture to output probabilities for each of the 4 possible actions.
2. Change the loss function appropriately (given that we are no longer giving a binary output)
3. Replace the play_one_step function with the following:


```def play_one_step(env, obs, model, loss_fn):
    with tf.GradientTape() as tape: # Record gradients without applying them
        action_probs = model(obs[np.newaxis]) # Use model policy to determine probability of each action
        action = tf.random.categorical(tf.math.log(action_probs), num_samples=1) # Use probability to decide action
        y_target = tf.one_hot(action, depth=4) # Define target as 1-action
        y_target = tf.reshape(y_target, action_probs.shape) # Match the shape of action_probs for loss computation
        loss = tf.reduce_mean(loss_fn(y_target, action_probs)) # Compute the gradient w.r.t. loss function

    grads = tape.gradient(loss, model.trainable_variables) # Save gradients for trainable variables
    obs, reward, done, truncated, info = env.step(int(action[0,0])) # Record environment state
    return obs, reward, done, truncated, grads # Return relevant information
```


4. Increase the number of simulations per update to 100 and reduce the iterations to 30

In [ ]:
n_iterations = 150
n_episodes_per_update = 10
n_max_steps = 200
discount_factor = 0.95

model = tf.keras.Sequential([
    tf.keras.layers.Dense(5, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

optimizer = tf.keras.optimizers.Nadam(learning_rate=0.01)
loss_fn = tf.keras.losses.binary_crossentropy

def play_one_step(env, obs, model, loss_fn):
    with tf.GradientTape() as tape: # Record gradients without applying them
        left_proba = model(obs[np.newaxis]) # Use model policy to determine probability of pushing left
        action = (tf.random.uniform([1, 1]) > left_proba) # Use probability to decide push left/right
        y_target = tf.constant([[1.]]) - tf.cast(action, tf.float32) # Define target as 1-action
            # (since action 0 means we went left, we want that to correspond to target of 100% going left)
        loss = tf.reduce_mean(loss_fn(y_target, left_proba)) # Compute the gradient w.r.t. loss function

    grads = tape.gradient(loss, model.trainable_variables) # Save gradients for trainable variables
    obs, reward, done, truncated, info = env.step(int(action)) # Record environment state
    return obs, reward, done, truncated, grads # Return relevant information

def play_multiple_episodes(env, n_episodes, n_max_steps, model, loss_fn):
    all_rewards = []
    all_grads = []
    for episode in range(n_episodes):
        current_rewards = []
        current_grads = []
        obs, info = env.reset()
        for step in range(n_max_steps):
            obs, reward, done, truncated, grads = play_one_step(
                env, obs, model, loss_fn)
            current_rewards.append(reward)
            current_grads.append(grads)
            if done or truncated:
                break

        all_rewards.append(current_rewards)
        all_grads.append(current_grads)

    return all_rewards, all_grads

def discount_rewards(rewards, discount_factor):
    discounted = np.array(rewards)
    for step in range(len(rewards) - 2, -1, -1):
        discounted[step] += discounted[step + 1] * discount_factor
    return discounted

def discount_and_normalize_rewards(all_rewards, discount_factor):
    all_discounted_rewards = [discount_rewards(rewards, discount_factor)
                              for rewards in all_rewards]
    flat_rewards = np.concatenate(all_discounted_rewards)
    reward_mean = flat_rewards.mean()
    reward_std = flat_rewards.std()
    return [(discounted_rewards - reward_mean) / reward_std
            for discounted_rewards in all_discounted_rewards]

In [ ]:
def train():
    for iteration in range(n_iterations):
        all_rewards, all_grads = play_multiple_episodes(
            env, n_episodes_per_update, n_max_steps, model, loss_fn)

        # extra code – displays some debug info during training
        total_rewards = sum(map(sum, all_rewards))
        print(f"\rIteration: {iteration + 1}/{n_iterations},"
              f" mean rewards: {total_rewards / n_episodes_per_update:.1f}", end="\n")

        all_final_rewards = discount_and_normalize_rewards(all_rewards,
                                                           discount_factor)
        all_mean_grads = []
        for var_index in range(len(model.trainable_variables)):
            mean_grads = tf.reduce_mean(
                [final_reward * all_grads[episode_index][step][var_index]
                 for episode_index, final_rewards in enumerate(all_final_rewards)
                     for step, final_reward in enumerate(final_rewards)], axis=0)
            all_mean_grads.append(mean_grads)

        optimizer.apply_gradients(zip(all_mean_grads, model.trainable_variables))

train()

In [ ]:
# extra code – displays the animation
def pg_policy(obs):
    action_probs = model.predict(obs[np.newaxis], verbose=0)
    action = tf.random.categorical(tf.math.log(action_probs), num_samples=1) # Use probability to decide action
    return int(action[0,0])

show_one_episode(pg_policy)

### Comment on the performance

### Modify the architecture of the neural model to allow the model to learn more complex behavior
#### Try a different number of layers and neurons per layer

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(5, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

optimizer = tf.keras.optimizers.Nadam(learning_rate=0.01)

train()

In [ ]:
# extra code – displays the animation
def pg_policy(obs):
    action_probs = model.predict(obs[np.newaxis], verbose=0)
    action = tf.random.categorical(tf.math.log(action_probs), num_samples=1) # Use probability to decide action
    return int(action[0,0])

show_one_episode(pg_policy)

### Your model likely learned to hover near the top of the viewing window.  Modify the play_multiple_episodes function to encourage the model to hover closer to the landing pad by changing its rewards.  Specifically, add 5 points to the score when the lander is within 0.5 of the ground (y-value only).

In [ ]:
def play_multiple_episodes(env, n_episodes, n_max_steps, model, loss_fn):
    all_rewards = []
    all_grads = []
    for episode in range(n_episodes):
        current_rewards = []
        current_grads = []
        obs, info = env.reset()
        for step in range(n_max_steps):
            obs, reward, done, truncated, grads = play_one_step(
                env, obs, model, loss_fn)
            # Modify the rewards



            current_rewards.append(reward)
            current_grads.append(grads)
            if done or truncated:
                break

        all_rewards.append(current_rewards)
        all_grads.append(current_grads)

    return all_rewards, all_grads

In [ ]:
old_model = model
model = tf.keras.models.clone_model(old_model)
optimizer = tf.keras.optimizers.Nadam(learning_rate=0.01)


train()

show_one_episode(pg_policy)

### Before submission, save the weights of your final model and submit along with this notebook.

In [ ]:
model.save_weights('model_weights.h5')